<a href="https://colab.research.google.com/github/nemitha21/S-P500_Stock_Analysis_CWK1/blob/main/S%26P500_Stock_Analysis_CWK1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import io, json, os, platform
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import requests
import yfinance as yf

# Folders (commit these to your public GitHub repo)
os.makedirs("data/raw", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

In [ ]:
START = "2021-01-01"
END = "2025-01-01"          # yfinance treats END as EXCLUSIVE, as the brief requires
BENCHMARK = "^GSPC"
WIKI_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

In [ ]:
RETRIEVAL_UTC = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

# Wikipedia can reject the default pandas user agent, so fetch the page first
html = requests.get(WIKI_URL, headers={"User-Agent": "Mozilla/5.0 (coursework; student)"}, timeout=30).text
tables = pd.read_html(io.StringIO(html))
constituents = tables[0].copy()          # first table = current constituents

constituents = constituents.rename(columns={
    "Symbol": "wiki_symbol",
    "Security": "company",
    "GICS Sector": "sector",
    "GICS Sub-Industry": "sub_industry",
    "Date added": "date_added",
})
# Yahoo uses "-" where Wikipedia uses "." (e.g. BRK.B -> BRK-B)
constituents["ticker"] = constituents["wiki_symbol"].str.replace(".", "-", regex=False)
constituents["retrieved_utc"] = RETRIEVAL_UTC

print("Retrieved:", RETRIEVAL_UTC)
print("Rows:", len(constituents), "| unique tickers:", constituents["ticker"].nunique())
print(constituents["sector"].value_counts())

constituents.to_csv("data/raw/sp500_constituents_wikipedia.csv", index=False)

Retrieved: 2026-11-18 15:22:48 UTC
Rows: 503 | unique tickers: 503
sector
Industrials               83
Information Technology    76
Financials                76
Health Care               60
Consumer Discretionary    47
Consumer Staples          33
Utilities                 31
Real Estate               30
Materials                 24
Communication Services    22
Energy                    21
Name: count, dtype: int64


In [ ]:
tickers = sorted(constituents["ticker"].unique().tolist())

# auto_adjust=False keeps BOTH Close and Adj Close (needed: adjusted for returns, Close for Monetary)
# actions=True adds Dividends and Stock Splits columns (needed for the corporate action checks in 1.2)
def download(tkrs):
    return yf.download(
        tkrs, start=START, end=END, interval="1d",
        auto_adjust=False, actions=True,
        group_by="column", threads=True, progress=False,
    )

raw = download(tickers + [BENCHMARK])
print(raw.shape)
print(raw.columns.get_level_values(0).unique().tolist())

ERROR:yfinance:
5 Failed downloads:
ERROR:yfinance:['SNDK', 'VYLR', 'HONA', 'FDXF', 'Q']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


(1005, 4022)
['Adj Close', 'Close', 'Dividends', 'High', 'Low', 'Open', 'Stock Splits', 'Volume']


In [ ]:
fields = ["Open", "High", "Low", "Close", "Adj Close", "Volume", "Dividends", "Stock Splits"]

def failed_list(df, tkrs):
    ok = df["Adj Close"].notna().any()
    return [t for t in tkrs if (t not in ok.index) or (not ok[t])]

failed = failed_list(raw, tickers + [BENCHMARK])
print("Failed first pass:", failed)

retry_log = []
for t in failed:
    try:
        d = download([t])
        if "Adj Close" in d.columns.get_level_values(0) and d["Adj Close"].notna().any().any():
            for f in fields:
                if f in d.columns.get_level_values(0):
                    raw[(f, t)] = d[f][t] if isinstance(d[f], pd.DataFrame) else d[f]
            retry_log.append((t, "recovered"))
        else:
            retry_log.append((t, "no data"))
    except Exception as e:
        retry_log.append((t, f"error: {e}"))

raw = raw.sort_index(axis=1)
still_failed = failed_list(raw, tickers + [BENCHMARK])
print("Retry log:", retry_log)
print("Still failed after retry:", still_failed)


Failed first pass: ['FDXF', 'HONA', 'Q', 'SNDK', 'VYLR']


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['FDXF']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['HONA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['Q']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['SNDK']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')
ERROR:yfinance:
1 Failed download:
ERRO

Retry log: [('FDXF', 'no data'), ('HONA', 'no data'), ('Q', 'no data'), ('SNDK', 'no data'), ('VYLR', 'no data')]
Still failed after retry: ['FDXF', 'HONA', 'Q', 'SNDK', 'VYLR']


In [ ]:
for f in fields:
    if f in raw.columns.get_level_values(0):
        raw[f].to_csv(f"data/raw/{f.replace(' ', '_').lower()}.csv")

# Benchmark on its own, for clarity
bench = pd.DataFrame({f: raw[f][BENCHMARK] for f in ["Close", "Adj Close", "Volume"]})
bench.to_csv("data/raw/gspc_benchmark.csv")

# Download log (feeds your inclusion/exclusion log in 1.2)
download_log = pd.DataFrame({
    "ticker": tickers + [BENCHMARK],
    "downloaded": [t not in still_failed for t in tickers + [BENCHMARK]],
})
download_log.to_csv("outputs/download_log.csv", index=False)

In [ ]:
meta = {
    "wikipedia_url": WIKI_URL,
    "retrieval_utc": RETRIEVAL_UTC,
    "data_source": "yfinance (Yahoo Finance), auto_adjust=False, actions=True",
    "start_inclusive": START,
    "end_exclusive": END,
    "benchmark": BENCHMARK,
    "n_tickers_requested": len(tickers),
    "n_tickers_failed": len(still_failed),
    "failed_tickers": still_failed,
    "first_date_in_data": str(raw.index.min().date()),
    "last_date_in_data": str(raw.index.max().date()),
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "yfinance": yf.__version__,
}
with open("data/raw/acquisition_metadata.json", "w") as fh:
    json.dump(meta, fh, indent=2)
print(json.dumps(meta, indent=2))

{
  "wikipedia_url": "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
  "retrieval_utc": "2026-11-18 15:22:48 UTC",
  "data_source": "yfinance (Yahoo Finance), auto_adjust=False, actions=True",
  "start_inclusive": "2021-01-01",
  "end_exclusive": "2025-01-01",
  "benchmark": "^GSPC",
  "n_tickers_requested": 503,
  "n_tickers_failed": 5,
  "failed_tickers": [
    "FDXF",
    "HONA",
    "Q",
    "SNDK",
    "VYLR"
  ],
  "first_date_in_data": "2021-01-04",
  "last_date_in_data": "2024-12-31",
  "python": "3.13.18",
  "pandas": "2.2.3",
  "numpy": "2.4.6",
  "yfinance": "0.2.66"
}


In [ ]:
print("Date range:", raw.index.min().date(), "to", raw.index.max().date())
print("Trading days:", len(raw))
print("Benchmark rows with Adj Close:", raw["Adj Close"][BENCHMARK].notna().sum())
print("Tickers with data:", raw["Adj Close"].notna().any().sum() - 1, "(excluding benchmark)")

Date range: 2021-01-04 to 2024-12-31
Trading days: 1005
Benchmark rows with Adj Close: 1005
Tickers with data: 498 (excluding benchmark)
